# Stress-Testing a Programmable Bank
**Elements of DeFi · New Banks · 45-minute interactive lab**

**Mission:** Defend a fictional crypto custodian against attacks **without blocking legitimate customers**.

**Start:** in JupyterLab select **Run → Run All Cells**. If the interface is missing, run `%pip install ipywidgets` in a new cell, restart the kernel, and choose Run All.


### Student checklist — track or print this for offline use

- [ ] **Baseline:** Challenge tab → score the weak starting bank.
- [ ] **Design:** Policy tab → choose your *own* controls to maximize the score while keeping routine payments usable.
- [ ] **Stress-test:** Incidents tab → predict and run **Crowbar, Mugging, Fraud, Lost device**. Save one observation for **each**; choose recovery participants for lost-device cases.
- [ ] **Improve:** Challenge tab → evaluate your revised policy. Try more combinations; keep your best design.
- [ ] **Code:** Complete the **Final coding exercise** below, run its cell, enable the rule in Policy, and verify it catches collusion without stopping a real supplier payment.
- [ ] **Submit:** Enter your baseline/best scores, incident observations, final policy, code, and reflection **directly into the course Google Form** (no notebook export).

<small>All attacks, keys, guardians, risk scores, and payments are simulated. This is a policy-design exercise, not a Fireblocks emulator. Based on real concepts from <a href="https://www.fireblocks.com/platforms/governance-and-policies">Fireblocks controls</a> and <a href="https://ncw-developers.fireblocks.com/docs/backup-recovery">protected key-share recovery</a>.</small>


In [ ]:
from dataclasses import dataclass, asdict, replace
from datetime import datetime, timezone
from pathlib import Path
from html import escape
import json, re

@dataclass(frozen=True)
class Policy:
    small_approvals: int = 1
    large_approvals: int = 2
    large_at: int = 5000
    tx_max: int = 25000
    daily_max: int = 30000
    destination_allowlist: bool = True
    new_address_delay_h: int = 24
    review_risk_at: int = 40
    block_risk_at: int = 80
    backup_enabled: bool = True
    recovery_guardians: int = 2
    offline_kit: bool = True
    admin_approvals: int = 2

@dataclass(frozen=True)
class Incident:
    key: str
    title: str
    kind: str
    intent: str
    detail: str
    question: str
    amount: int = 0
    available_approvals: int = 0
    allowlisted: bool = True
    payee_age_h: int = 999
    risk: int = 0
    already_spent: int = 0
    after_hours: bool = False
    passphrase_known: bool = False
    recovery_guardians_present: int = 0
    provider_available: bool = True
    offline_trustees: int = 0

INCIDENTS = [
    Incident('crowbar','Crowbar: coerced treasurer','transfer','attack',
             'One employee is coerced to send $40,000 to an existing approved vendor address. The attacker controls only that person’s approval.',
             'Does a valid signer alone have the authority to send?',40000,1,True,999,4),
    Incident('mugging','Mugging: unlocked phone','transfer','attack',
             'A thief has a signed-in device and tries a $650 transfer to an unfamiliar address created one hour ago.',
             'Which rule can stop this even though the device is unlocked?',650,1,False,1,5),
    Incident('fraud','Fraud: suspicious payee','transfer','attack',
             'A $9,200 transfer has two approvals and uses a known payee, but the monitoring provider reports risk 95/100.',
             'Does the transaction policy still stop it?',9200,2,True,999,95),
    Incident('lost_device','Key loss: real owner','recovery','recovery',
             'The real owner has lost their signing device. Choose whether the recovery secret and independent guardian approvals are available. A protected backup is needed.',
             'Can the owner restore signing authority?',passphrase_known=True,recovery_guardians_present=2),
    Incident('recovery_attack','Recovery: impersonator','recovery','attack',
             'An impostor attempts to use a stolen recovery secret. Choose which independent guardians, if any, also participate or are compromised.',
             'Can a stolen recovery secret alone restore access?',passphrase_known=True,recovery_guardians_present=0),
    Incident('admin_attack','Admin: weaken the rules','admin','attack',
             'A compromised admin tries to lower the approval threshold and disable the allowlist using only their own admin approval.',
             'Can one administrator weaken the policy?',available_approvals=1),
    Incident('small_payment','Everyday: coffee + supplies','transfer','legit',
             'A normal $350 payment to an established supplier. One genuine signer is present; low risk.',
             'Should this be immediate?',350,1,True,999,3),
    Incident('large_payment','Everyday: major supplier','transfer','legit',
             'A genuine $18,000 payment to an established supplier, with two independent approvals.',
             'Can the business pay an important supplier?',18000,2,True,999,4),
    Incident('new_payee','Everyday: new vendor','transfer','legit',
             'A genuine $900 invoice to a new vendor verified 72 hours ago. It is not yet on the formal address allowlist.',
             'Should a verified new vendor always require manual work?',900,1,False,72,12),
    Incident('urgent_payment','Everyday: urgent repair','transfer','legit',
             'A genuine $4,200 emergency repair payment; $4,000 has already been sent today.',
             'Can the firm make an urgent payment?',4200,1,True,999,5,4000),
    Incident('false_positive','Everyday: false alarm','transfer','legit',
             'A legitimate $200 merchant payment is given a noisy monitoring score of 50/100.',
             'How much legitimate activity is delayed by screening?',200,1,True,999,50),
    Incident('admin_legit','Admin: legitimate update','admin','legit',
             'Two administrators approve a normal policy update.',
             'Does normal governance still function?',available_approvals=2),
    Incident('provider_outage','Recovery: provider outage','recovery','recovery',
             'The custody provider is unavailable. Choose which independent offline trustees cooperate; an independently prepared recovery kit is also necessary.',
             'Is there an independently prepared way to exit?',provider_available=False,offline_trustees=2),
    Incident('collusion','Advanced: full-quorum collusion','transfer','edge',
             'A complete authorized quorum approves $18,000 to a known destination with a low-risk score, but at an unusual hour. Ordinary policy inputs otherwise resemble a genuine supplier payment.',
             'What new signal would distinguish this from a legitimate payment?',18000,2,True,999,4,0,True),
]
BY_KEY={x.key:x for x in INCIDENTS}
REQUIRED_KEYS=['crowbar','mugging','fraud','lost_device']
BENCH_KEYS=['small_payment','large_payment','new_payee','urgent_payment','false_positive',
            'admin_legit','crowbar','mugging','fraud','recovery_attack','admin_attack',
            'lost_device','provider_outage']

@dataclass
class Decision:
    incident: str
    outcome: str
    why: str
    trace: list

def _step(trace, label, status, detail):
    trace.append({'stage':label,'status':status,'detail':detail})

def evaluate(policy, incident, extra_rule=None):
    """Pure, deterministic simulation. No cryptography or real funds are involved."""
    tr=[]
    if incident.kind=='admin':
        req=policy.admin_approvals
        _step(tr,'Admin change quorum', 'PASS' if incident.available_approvals>=req else 'STOP',
              f'{incident.available_approvals} independent admins available; {req} required')
        good=incident.available_approvals>=req
        return Decision(incident.key,'APPROVED' if good else 'DENIED',
                        'Authorized admin quorum' if good else 'One admin cannot change protected policy',tr)
    if incident.kind=='recovery':
        if not incident.provider_available:
            ok=policy.offline_kit and incident.offline_trustees>=2
            _step(tr,'Independent continuity kit','PASS' if ok else 'STOP',
                  f'Kit configured: {policy.offline_kit}; offline trustees: {incident.offline_trustees}/2')
            return Decision(incident.key,'EXIT_AVAILABLE' if ok else 'UNAVAILABLE',
                            'Offline recovery path is prepared, not executed' if ok else 'No usable independent recovery path',tr)
        if not policy.backup_enabled:
            _step(tr,'Encrypted key-share backup','STOP','No backup was created before device loss')
            return Decision(incident.key,'UNAVAILABLE','No protected share available for restoration',tr)
        _step(tr,'Encrypted key-share backup','PASS','Encrypted share is available (not a plaintext private key)')
        if not incident.passphrase_known:
            _step(tr,'Recovery secret','STOP','Cannot decrypt the protected share')
            return Decision(incident.key,'DENIED','Missing the recovery secret',tr)
        _step(tr,'Recovery secret','PASS','Correct decryption secret presented')
        if incident.recovery_guardians_present<policy.recovery_guardians:
            _step(tr,'Recovery guardian check','STOP',
                  f'{incident.recovery_guardians_present} independent guardians; {policy.recovery_guardians} required')
            return Decision(incident.key,'DENIED','Insufficient independent recovery authorization',tr)
        _step(tr,'Recovery guardian check','PASS',
              f'{incident.recovery_guardians_present} available; {policy.recovery_guardians} required')
        _step(tr,'Recovery action','PASS','Signing access restored in the simulator')
        return Decision(incident.key,'RESTORED','Backup + secret + recovery authorizations satisfied',tr)
    # Transaction workflow: no signing if any pre-signing control blocks it.
    if incident.risk>=policy.block_risk_at:
        _step(tr,'Monitoring / screening','STOP',f'Risk {incident.risk} >= block threshold {policy.block_risk_at}')
        return Decision(incident.key,'BLOCKED','Monitoring block threshold',tr)
    if incident.risk>=policy.review_risk_at:
        _step(tr,'Monitoring / screening','HOLD',f'Risk {incident.risk} >= review threshold {policy.review_risk_at}')
        return Decision(incident.key,'REVIEW','Flagged for human review; not broadcast',tr)
    _step(tr,'Monitoring / screening','PASS',f'Risk {incident.risk} below review threshold')
    if policy.destination_allowlist and not incident.allowlisted:
        _step(tr,'Destination policy','STOP','Address not on allowlist')
        return Decision(incident.key,'BLOCKED','Destination allowlist',tr)
    if not incident.allowlisted and incident.payee_age_h<policy.new_address_delay_h:
        _step(tr,'Destination cooldown','HOLD',
              f'Payee age {incident.payee_age_h} h < wait {policy.new_address_delay_h} h')
        return Decision(incident.key,'PENDING','New payee cooldown; no signing',tr)
    _step(tr,'Destination policy','PASS',
          'Allowlisted destination' if incident.allowlisted else 'New destination passed configured checks')
    if incident.amount>policy.tx_max:
        _step(tr,'Single-transfer limit','STOP',f'${incident.amount:,} > cap ${policy.tx_max:,}')
        return Decision(incident.key,'BLOCKED','Single transfer exceeds limit',tr)
    if incident.amount+incident.already_spent>policy.daily_max:
        _step(tr,'Daily outflow limit','STOP',
              f'${incident.amount:,} + ${incident.already_spent:,} > daily cap ${policy.daily_max:,}')
        return Decision(incident.key,'BLOCKED','Daily outflow cap',tr)
    _step(tr,'Limits','PASS',
          f'${incident.amount:,} transaction; cumulative ${incident.amount+incident.already_spent:,}')
    if extra_rule is not None:
        try:
            action=extra_rule(incident,policy)
        except Exception as exc:
            _step(tr,'Student custom rule','ERROR',f'{type(exc).__name__}: {str(exc)[:120]}')
            return Decision(incident.key,'ERROR','Student custom rule raised an exception',tr)
        if action not in (None,'BLOCK','REVIEW'):
            _step(tr,'Student custom rule','ERROR','Rule must return None, "BLOCK", or "REVIEW"')
            return Decision(incident.key,'ERROR','Invalid rule output',tr)
        if action is not None:
            _step(tr,'Student custom rule','STOP' if action=='BLOCK' else 'HOLD',f'Returned {action}')
            return Decision(incident.key,'BLOCKED' if action=='BLOCK' else 'REVIEW','Custom risk control',tr)
        _step(tr,'Student custom rule','PASS','No exception rule matched')
    required=policy.large_approvals if incident.amount>=policy.large_at else policy.small_approvals
    if incident.available_approvals<required:
        _step(tr,'Independent approvals','HOLD',
              f'{incident.available_approvals} present; {required} required')
        return Decision(incident.key,'PENDING','Approval quorum not met; no signing',tr)
    _step(tr,'Independent approvals','PASS',
          f'{incident.available_approvals} independent approvals satisfy required {required}')
    _step(tr,'MPC signing','PASS','Signature generated only after policy approval (not simulated cryptographically)')
    _step(tr,'Ledger settlement','PASS','Simulated token balance transfer; no real chain used')
    return Decision(incident.key,'EXECUTED','All controls passed',tr)

SAFE_NONEXEC={'BLOCKED','PENDING','REVIEW','DENIED','UNAVAILABLE'}
CORE_ATTACKS=['crowbar','mugging','fraud','recovery_attack','admin_attack']
CORE_LEGIT=['small_payment','large_payment','new_payee','urgent_payment','false_positive','admin_legit']
CORE_RECOVERY=['lost_device','provider_outage']

def benchmark(policy, rule=None):
    results={k:evaluate(policy,BY_KEY[k],rule) for k in BENCH_KEYS}
    secure=sum(results[k].outcome in SAFE_NONEXEC for k in CORE_ATTACKS)
    usable=sum(results[k].outcome==('APPROVED' if k=='admin_legit' else 'EXECUTED') for k in CORE_LEGIT)
    resilient=sum(results[k].outcome==('RESTORED' if k=='lost_device' else 'EXIT_AVAILABLE') for k in CORE_RECOVERY)
    pct_security=100*secure/len(CORE_ATTACKS)
    pct_usability=100*usable/len(CORE_LEGIT)
    pct_resilience=100*resilient/len(CORE_RECOVERY)
    weighted=round(.5*pct_security+.3*pct_usability+.2*pct_resilience,1)
    advanced=evaluate(policy,BY_KEY['collusion'],rule)
    return {'results':results, 'counts':{'security':(secure,len(CORE_ATTACKS)),
                                        'usability':(usable,len(CORE_LEGIT)),
                                        'recovery':(resilient,len(CORE_RECOVERY))},
            'percent':{'Security':pct_security,'Usability':pct_usability,'Recovery':pct_resilience},
            'score':weighted,'advanced':advanced}

DEFAULT_POLICY=Policy()
# Self-tests: run automatically at notebook startup to detect accidental edits.
def self_tests():
    import dataclasses
    p=DEFAULT_POLICY
    assert evaluate(p,BY_KEY['small_payment']).outcome=='EXECUTED'
    assert evaluate(p,BY_KEY['large_payment']).outcome=='EXECUTED'
    assert evaluate(p,BY_KEY['crowbar']).outcome=='BLOCKED'
    assert evaluate(p,BY_KEY['mugging']).outcome=='BLOCKED'
    assert evaluate(p,BY_KEY['fraud']).outcome=='BLOCKED'
    assert evaluate(p,BY_KEY['lost_device']).outcome=='RESTORED'
    assert evaluate(p,BY_KEY['recovery_attack']).outcome=='DENIED'
    assert evaluate(p,BY_KEY['provider_outage']).outcome=='EXIT_AVAILABLE'
    assert evaluate(p,BY_KEY['admin_attack']).outcome=='DENIED'
    assert evaluate(p,BY_KEY['admin_legit']).outcome=='APPROVED'
    assert evaluate(p,BY_KEY['collusion']).outcome=='EXECUTED' # intentional baseline blind spot
    no_q=dataclasses.replace(p,large_approvals=1,tx_max=100000,daily_max=100000)
    assert evaluate(no_q,BY_KEY['crowbar']).outcome=='EXECUTED'
    lost=dataclasses.replace(p,backup_enabled=False)
    assert evaluate(lost,BY_KEY['lost_device']).outcome=='UNAVAILABLE'
    no_guard=dataclasses.replace(p,recovery_guardians=0)
    assert evaluate(no_guard,BY_KEY['recovery_attack']).outcome=='RESTORED'
    no_admin=dataclasses.replace(p,admin_approvals=1)
    assert evaluate(no_admin,BY_KEY['admin_attack']).outcome=='APPROVED'
    permissive=dataclasses.replace(p,destination_allowlist=False,new_address_delay_h=0,
                                  large_approvals=1,tx_max=100000,daily_max=100000,
                                  review_risk_at=100,block_risk_at=101)
    assert evaluate(permissive,BY_KEY['mugging']).outcome=='EXECUTED'
    assert evaluate(permissive,BY_KEY['fraud']).outcome=='EXECUTED'
    assert evaluate(p,BY_KEY['false_positive']).outcome=='REVIEW'
    assert evaluate(p,BY_KEY['new_payee']).outcome=='BLOCKED'
    def unusual(tx,policy): return 'REVIEW' if tx.after_hours and tx.amount>=15000 else None
    assert evaluate(p,BY_KEY['collusion'],unusual).outcome=='REVIEW'
    good=dataclasses.replace(p,destination_allowlist=False,review_risk_at=60)
    x=benchmark(good)
    assert x['score']==100, x
    assert evaluate(good,BY_KEY['mugging']).outcome=='PENDING'
    assert benchmark(p)['score']<100
    assert evaluate(p,BY_KEY['small_payment'],lambda tx, p:'bad').outcome=='ERROR'
    assert evaluate(p,BY_KEY['small_payment'],lambda tx, p:1/0).outcome=='ERROR'
    return 'Simulation checks passed (25 assertions).'
ENGINE_STATUS=self_tests()
print(ENGINE_STATUS)

In [ ]:
from IPython.display import display, HTML
import hashlib
try:
    import ipywidgets as W
except ImportError as exc:
    raise RuntimeError('Interactive controls require ipywidgets. In a new cell, run: %pip install ipywidgets ; then restart the kernel and Run All.') from exc

STYLE='''<style>
.banklab {font-family:system-ui,Arial,sans-serif;line-height:1.45;color:var(--jp-ui-font-color1,var(--vscode-foreground,#1f2937))}
.banklab .head {font-size:18px;font-weight:700;margin:0 0 4px}
.banklab .sub {font-size:12px;color:var(--jp-ui-font-color2,var(--vscode-descriptionForeground,#667085));margin-bottom:9px}
.banklab .panel {padding:12px 14px;border:1px solid #dedfe2;border-radius:9px;margin:7px 0;background:#fff;color:#1f2937}
.banklab .label {color:#667085;font-size:12px;text-transform:uppercase;letter-spacing:.04em;font-weight:600}
.banklab .strong {font-weight:700;font-size:17px}
.banklab .pill {display:inline-block;border:1px solid #d7dbe1;border-radius:15px;padding:3px 8px;margin:3px 5px 3px 0;font-size:12px;background:#f7f8fa}
.banklab .ok{color:#176641}.banklab .bad{color:#a52a28}.banklab .hold{color:#795b17}
.banklab .trace{border-left:3px solid #d7dbe1;padding:7px 10px;margin:3px 0;background:#f8fafc;font-size:13px}
.banklab .bar-bg{height:12px;border-radius:9px;background:#e9edf1;overflow:hidden;margin:3px 0 10px}
.banklab .bar-in{height:12px;background:#6b859e;border-radius:9px}
.banklab table{border-collapse:collapse;width:100%;font-size:12px}
.banklab th,.banklab td{border-bottom:1px solid #e8e9eb;padding:7px;text-align:left}
.banklab .small{font-size:12px;color:#667085}
.banklab .result{font-size:19px;font-weight:750;margin:5px 0}
</style>'''
display(HTML(STYLE))

history=[]
notes={}
last_run_by_incident={}
suite_runs=[]
last_output=None

small_app=W.Dropdown(options=[('1 approval',1),('2 approvals',2)],value=1,description='Routine tx:',style={'description_width':'145px'},layout=W.Layout(width='370px'))
large_app=W.Dropdown(options=[('1 approval',1),('2 approvals',2),('3 approvals',3)],value=1,description='Large tx:',style={'description_width':'145px'},layout=W.Layout(width='370px'))
large_at=W.Dropdown(options=[('$2,000',2000),('$5,000',5000),('$10,000',10000),('$25,000',25000)],value=5000,description='Large starts at:',style={'description_width':'145px'},layout=W.Layout(width='370px'))
max_tx=W.Dropdown(options=[('$5,000',5000),('$10,000',10000),('$25,000',25000),('$100,000',100000)],value=100000,description='Per transfer:',style={'description_width':'145px'},layout=W.Layout(width='370px'))
daily_max=W.Dropdown(options=[('$10,000',10000),('$30,000',30000),('$100,000',100000)],value=100000,description='Per day:',style={'description_width':'145px'},layout=W.Layout(width='370px'))
allowlist=W.Checkbox(value=False,description='Only allowlisted destinations',indent=False,layout=W.Layout(width='365px'))
cooldown=W.Dropdown(options=[('No delay',0),('24 hours',24),('48 hours',48)],value=0,description='New-payee delay:',style={'description_width':'145px'},layout=W.Layout(width='370px'))
review=W.Dropdown(options=[('40 / 100',40),('50 / 100',50),('60 / 100',60),('80 / 100',80),('100 / 100 (off)',100)],value=100,description='Review risk ≥',style={'description_width':'145px'},layout=W.Layout(width='370px'))
block=W.Dropdown(options=[('80 / 100',80),('90 / 100',90),('100 / 100',100),('101 / 100 (off)',101)],value=101,description='Block risk ≥',style={'description_width':'145px'},layout=W.Layout(width='370px'))
backup=W.Checkbox(value=False,description='Encrypted key-share backup pre-created',indent=False,layout=W.Layout(width='375px'))
guardians=W.Dropdown(options=[('0 (unsafe)',0),('1 guardian',1),('2 guardians',2)],value=0,description='Recovery quorum:',style={'description_width':'145px'},layout=W.Layout(width='370px'))
kit=W.Checkbox(value=False,description='Independent break-glass recovery kit pre-created',indent=False,layout=W.Layout(width='420px'))
admin=W.Dropdown(options=[('1 (unsafe)',1),('2 admins',2),('3 admins',3)],value=1,description='Policy edits need:',style={'description_width':'145px'},layout=W.Layout(width='370px'))
use_rule=W.Checkbox(value=False,description='Use my coding-exercise rule (see code cell below)',indent=False,layout=W.Layout(width='490px'))

controls=[small_app,large_app,large_at,max_tx,daily_max,allowlist,cooldown,review,block,backup,guardians,kit,admin,use_rule]

def get_policy():
    return Policy(small_app.value,large_app.value,large_at.value,max_tx.value,daily_max.value,
                  allowlist.value,cooldown.value,review.value,block.value,
                  backup.value,guardians.value,kit.value,admin.value)

def get_rule():
    if not use_rule.value:
        return None
    fn=globals().get('student_rule')
    return fn if callable(fn) else (lambda tx,policy:'Invalid student_rule: run the coding exercise cell')

policy_view=W.HTML()
transfer_policy=W.VBox([W.HTML('<b>Authorization & spending</b>'),small_app,large_app,large_at,max_tx,daily_max,
                        W.HTML('<b>Destinations & screening</b>'),allowlist,cooldown,review,block])
recovery_policy=W.VBox([
    W.HTML('<b>Recovery & governance</b>'),
    W.HTML('<div style="max-width:520px;padding:9px;border-left:3px solid #6b859e;background:#edf2f8;color:#1a2940">'
           '<b>Who are recovery guardians?</b> Independent people who approve restoration of a lost signing key share. '
           'They are <b>not</b> the original signer and <b>not</b> the recovery passphrase holder. '
           'The quorum below specifies how many must approve; choose the participants for each recovery attempt on the Incidents tab. '
           '<br><small>This additional guardian gate is a <b>lab policy choice</b>, not a universal Fireblocks product requirement.</small></div>'),
    backup,guardians,kit,admin,use_rule])
policy_acc=W.Accordion(children=[transfer_policy,recovery_policy],selected_index=0)
policy_acc.set_title(0,'Transaction rules')
policy_acc.set_title(1,'Recovery + admin rules')
policy_tab=W.VBox([W.HTML('<div class="banklab"><div class="head">1 · Set the bank policy</div><div class="sub">Start with a weak bank. Choose your own controls, then use Challenge to maximize the score without blocking good payments.</div></div>'),
                   policy_acc,policy_view])

incident_opts=[(x.title,x.key) for x in INCIDENTS]
choose=W.Dropdown(options=incident_opts,value='crowbar',description='Scenario:',style={'description_width':'85px'},layout=W.Layout(width='560px'))
predict=W.Dropdown(description='Predict:',options=[('Select first','')],style={'description_width':'85px'},layout=W.Layout(width='390px'))
run_btn=W.Button(description='▶ Run incident',button_style='primary',layout=W.Layout(width='165px'))
incident_context=W.HTML()
# These choices modify ONLY the currently run recovery incident. The benchmark retains a fixed suite.
recovery_secret=W.Checkbox(description='Recovery secret is presented (by owner OR attacker)',
                           indent=False,layout=W.Layout(width='520px'))
recovery_finance=W.Checkbox(description='Guardian: Finance approver',indent=False,layout=W.Layout(width='320px'))
recovery_security=W.Checkbox(description='Guardian: Security approver',indent=False,layout=W.Layout(width='320px'))
recovery_operations=W.Checkbox(description='Guardian: Operations approver',indent=False,layout=W.Layout(width='320px'))
trustee_finance=W.Checkbox(description='Offline trustee: Finance',indent=False,layout=W.Layout(width='320px'))
trustee_security=W.Checkbox(description='Offline trustee: Security',indent=False,layout=W.Layout(width='320px'))
trustee_operations=W.Checkbox(description='Offline trustee: Operations',indent=False,layout=W.Layout(width='320px'))
recovery_participant_summary=W.HTML()
recovery_participants=W.VBox([])
recovery_widgets=[recovery_secret,recovery_finance,recovery_security,recovery_operations,
                  trustee_finance,trustee_security,trustee_operations]
loading_participants=False
participant_states={}
for incident in INCIDENTS:
    if incident.kind!='recovery': continue
    participant_states[incident.key]={
        'secret':incident.passphrase_known,
        'finance':incident.recovery_guardians_present>=1,
        'security':incident.recovery_guardians_present>=2,
        'operations':incident.recovery_guardians_present>=3,
        'trustee_finance':incident.offline_trustees>=1,
        'trustee_security':incident.offline_trustees>=2,
        'trustee_operations':incident.offline_trustees>=3,
    }

trace_view=W.HTML()
observation=W.Textarea(placeholder='One sentence: which policy mattered, or what failed?',
                       description='Your note:',style={'description_width':'85px'},layout=W.Layout(width='680px',height='72px'))
save_note_btn=W.Button(description='Manually paste observation to form',button_style='',layout=W.Layout(width='295px'))
note_msg=W.HTML()
scenario_tab=W.VBox([W.HTML('<div class="banklab"><div class="head">2 · Run an incident</div><div class="sub">Pick an event, predict what happens, then run it. Change policy and rerun to test your hypothesis.</div></div>'),
                      choose,incident_context,recovery_participants,W.HBox([predict,run_btn]),trace_view,observation,W.HBox([save_note_btn,note_msg])])

suite_btn=W.Button(description='▶ Evaluate current bank',button_style='primary',layout=W.Layout(width='220px'))
suite_view=W.HTML(value='<div class="banklab small">Run once with the unsafe starting policy, then rerun after improving it.</div>')
score_tab=W.VBox([W.HTML('<div class="banklab"><div class="head">3 · Design your best bank policy</div><div class="sub">Your goal: maximize the 0–100 score by changing policies yourself. Security = 50%, usability = 30%, recovery = 20%. Blocking every payment will score poorly.</div></div>'),
                  suite_btn,suite_view])

tabs=W.Tab(children=[policy_tab,scenario_tab,score_tab])
for idx,title in enumerate(['Policy','Incidents','Challenge']): tabs.set_title(idx,title)


def status_class(status):
    if status in ('EXECUTED','RESTORED','APPROVED','EXIT_AVAILABLE','PASS'):
        return 'ok'
    if status in ('BLOCKED','DENIED','UNAVAILABLE','ERROR','STOP'):
        return 'bad'
    return 'hold'

def preview_policy(*_):
    p=get_policy()
    policy_view.value=(f'<div class="banklab panel"><div class="label">Current design</div>'
                       f'<div class="small">Routine: {p.small_approvals} signers; large ≥ ${p.large_at:,}: {p.large_approvals}; '
                       f'transfer cap ${p.tx_max:,}; daily ${p.daily_max:,}. '
                       f'Allowlist: {"ON" if p.destination_allowlist else "OFF"}; '
                       f'recovery guardians: {p.recovery_guardians}.</div></div>')
    refresh_recovery_participant_summary()


def selected_recovery_participants(s):
    if s.kind!='recovery': return None
    if not s.provider_available:
        names=[label for label,widget in [('Finance',trustee_finance),('Security',trustee_security),
                                          ('Operations',trustee_operations)] if widget.value]
        return {'mode':'offline-exit','offline_trustees':names,'count':len(names)}
    names=[label for label,widget in [('Finance',recovery_finance),('Security',recovery_security),
                                      ('Operations',recovery_operations)] if widget.value]
    return {'mode':'lost-device','recovery_secret_presented':bool(recovery_secret.value),
            'guardians':names,'count':len(names)}


def chosen_incident():
    s=BY_KEY[choose.value]
    participants=selected_recovery_participants(s)
    if participants is None: return s
    if s.provider_available:
        return replace(s,passphrase_known=participants['recovery_secret_presented'],
                       recovery_guardians_present=participants['count'])
    return replace(s,offline_trustees=participants['count'])


def refresh_recovery_participant_summary():
    if choose.value not in BY_KEY: return
    s=BY_KEY[choose.value]
    participants=selected_recovery_participants(s)
    if participants is None: return
    if s.provider_available:
        need=get_policy().recovery_guardians
        message=(f'Selected <b>{participants["count"]}</b> independent guardian(s); '
                 f'policy requires <b>{need}</b>. '
                 'The decryption secret does not count as a guardian approval.')
    else:
        message=(f'Selected <b>{participants["count"]}</b> offline trustee(s); '
                 'independent exit needs <b>2</b> plus a pre-created offline kit. '
                 'This is separate from lost-device guardian recovery.')
    recovery_participant_summary.value=(
        '<div style="padding:7px;background:#edf2f8;color:#1a2940;max-width:640px">'
        +message+'</div>')


def update_recovery_participant_settings():
    global loading_participants
    s=BY_KEY[choose.value]
    if s.kind!='recovery':
        recovery_participants.children=()
        return
    state=participant_states[s.key]
    loading_participants=True
    try:
        for key,widget in [
            ('secret',recovery_secret),('finance',recovery_finance),
            ('security',recovery_security),('operations',recovery_operations),
            ('trustee_finance',trustee_finance),('trustee_security',trustee_security),
            ('trustee_operations',trustee_operations)]:
            widget.value=state[key]
    finally:
        loading_participants=False
    if s.provider_available:
        recovery_participants.children=(
            W.HTML('<div style="font-weight:700;margin-top:8px;color:#1a2940;background:#f7f9fc;padding:6px">Who participated in this recovery attempt?</div>'
                   '<div style="font-size:12px">Select the decryption-secret holder and each <b>independent</b> guardian. '
                   'For an impersonator, checking a guardian simulates that guardian also cooperating or being compromised.</div>'),
            recovery_secret,recovery_finance,recovery_security,recovery_operations,
            recovery_participant_summary)
    else:
        recovery_participants.children=(
            W.HTML('<div style="font-weight:700;margin-top:8px;color:#1a2940;background:#f7f9fc;padding:6px">Who participated in the offline recovery?</div>'
                   '<div style="font-size:12px">Select independent offline trustees. This path requires a '
                   'pre-created break-glass kit; the provider is unavailable.</div>'),
            trustee_finance,trustee_security,trustee_operations,recovery_participant_summary)
    refresh_recovery_participant_summary()


def participant_changed(change=None):
    if loading_participants: return
    s=BY_KEY[choose.value]
    if s.kind!='recovery': return
    participant_states[s.key]={
        'secret':bool(recovery_secret.value),'finance':bool(recovery_finance.value),
        'security':bool(recovery_security.value),'operations':bool(recovery_operations.value),
        'trustee_finance':bool(trustee_finance.value),'trustee_security':bool(trustee_security.value),
        'trustee_operations':bool(trustee_operations.value)}
    refresh_recovery_participant_summary()
    trace_view.value=''
    note_msg.value='<span class="banklab small">Participants changed. Predict and rerun to save this outcome.</span>'


def update_context(change=None):
    s=BY_KEY[choose.value]
    prediction_options={'transfer':[('Select first',''),('Execute','EXECUTED'),('Block','BLOCKED'),
                                     ('Pending approval / cooldown','PENDING'),('Human review','REVIEW')],
                        'recovery':[('Select first',''),('Restore access','RESTORED'),
                                    ('Deny recovery','DENIED'),('Recovery unavailable','UNAVAILABLE'),
                                    ('External exit available','EXIT_AVAILABLE')],
                        'admin':[('Select first',''),('Approve change','APPROVED'),('Deny change','DENIED')]}
    predict.options=prediction_options[s.kind]
    predict.value=''
    # A light, explicitly colored description panel remains legible in dark Jupyter themes.
    scenario_label='Attack description' if s.intent=='attack' else 'Scenario description'
    desc=(f'<div class="banklab panel" style="background:#f7f9fc;color:#1a2940;max-width:720px">'
          f'<div style="font-weight:700;font-size:16px;color:#1a2940">{escape(s.title)}</div>'
          f'<div style="font-size:11px;color:#365979;text-transform:uppercase;margin-top:6px">{scenario_label}</div>'
          f'<div style="color:#1a2940;font-size:14px">{escape(s.detail)}</div>')
    if s.kind=='transfer':
        desc+=(f'<div style="margin-top:7px"><span class="pill">${s.amount:,}</span>'
               f'<span class="pill">{s.available_approvals} signer(s)</span>'
               f'<span class="pill">{"Allowlisted" if s.allowlisted else "New payee"}</span>'
               f'<span class="pill">Risk {s.risk}/100</span></div>')
    desc+=f'<div style="font-size:13px;margin-top:8px;color:#1a2940"><b>Question:</b> {escape(s.question)}</div></div>'
    incident_context.value=desc
    update_recovery_participant_settings()
    trace_view.value=''
    observation.value=notes.get(s.key,'')
    note_msg.value=''


def run_incident(_):
    if not predict.value:
        trace_view.value='<div class="banklab panel bad">Choose a prediction before running the incident.</div>'
        return
    s=chosen_incident()
    policy=get_policy()
    result=evaluate(policy,s,get_rule())
    participant_snapshot=selected_recovery_participants(s)
    event={'time_utc':datetime.now(timezone.utc).isoformat(timespec='seconds'),
           'incident':s.key,'title':s.title,'kind':s.kind,'prediction':predict.value,
           'outcome':result.outcome,'prediction_correct':predict.value==result.outcome,
           'reason':result.why,'trace':result.trace,'policy':asdict(policy),
           'custom_rule_enabled':bool(use_rule.value),'observation':notes.get(s.key,''),
           'recovery_participants':participant_snapshot}
    history.append(event)
    last_run_by_incident[s.key]=event
    state='Correct prediction' if event['prediction_correct'] else 'Different from prediction'
    trace=''.join(f'<div class="trace"><b>{escape(t["stage"])}</b> — <b class="{status_class(t["status"])}">{escape(t["status"])}</b> · {escape(t["detail"])}</div>' for t in result.trace)
    trace_view.value=(f'<div class="banklab panel"><div class="label">Result</div>'
                      f'<div class="result {status_class(result.outcome)}">{escape(result.outcome)}</div>'
                      f'<div class="small">{escape(state)} · {escape(result.why)}</div>{trace}</div>')


def save_observation(_):
    k=choose.value
    if k not in last_run_by_incident:
        note_msg.value='<span class="banklab bad small">Run this incident first.</span>'; return
    s=BY_KEY[k]
    if s.kind=='recovery' and last_run_by_incident[k].get('recovery_participants')!=selected_recovery_participants(s):
        note_msg.value='<span class="banklab bad small">Recovery participants changed; rerun the incident before saving.</span>'
        return
    note=observation.value.strip()
    if not note:
        note_msg.value='<span class="banklab bad small">Write one short observation.</span>'; return
    notes[k]=note
    last_run_by_incident[k]['observation']=note
    note_msg.value='<span class="banklab ok small">Copy this observation into your Google Form. Not submitted automatically.</span>'


def render_score(stats):
    r=stats['results']
    cols=[]
    for label,pct in stats['percent'].items():
        good,tot=stats['counts'][label.lower()]
        cols.append(f'<div style="flex:1;min-width:155px"><div class="label">{escape(label)}</div>'
                    f'<div class="strong">{good}/{tot} · {pct:.0f}%</div>'
                    f'<div class="bar-bg"><div class="bar-in" style="width:{pct:.1f}%"></div></div></div>')
    table=''.join(f'<tr><td>{escape(BY_KEY[k].title)}</td><td>{"Legit" if k in CORE_LEGIT else ("Recover" if k in CORE_RECOVERY else "Attack")}</td>'
                  f'<td class="{status_class(r[k].outcome)}"><b>{r[k].outcome}</b></td></tr>' for k in BENCH_KEYS)
    advanced=stats['advanced']
    return (f'<div class="banklab"><div class="panel"><div class="label">Weighted score</div>'
            f'<div class="strong">{stats["score"]:.1f} / 100</div>'
            f'<div style="display:flex;gap:16px;flex-wrap:wrap;margin-top:12px">{"".join(cols)}</div></div>'
            f'<div class="panel"><b>Core test outcomes</b><table><tr><th>Scenario</th><th>Type</th><th>Result</th></tr>{table}</table></div>'
            f'<div class="panel"><b>Blind spot (not scored):</b> full-quorum collusion → '
            f'<b class="{status_class(advanced.outcome)}">{escape(advanced.outcome)}</b>. '
            f'Can your coding-exercise rule flag an unusual-hour transfer without blocking legitimate activity?</div>'
            f'<div class="small">Non-execution counts as stopping an attack; human reviews and pending approvals count as friction for legitimate transactions. This is a teaching score, not a security audit.</div></div>')


def rule_fingerprint():
    if not use_rule.value:
        return None
    fn=globals().get('student_rule')
    if not callable(fn):
        return 'NOT_DEFINED'
    code=fn.__code__
    data=code.co_code+repr(code.co_consts).encode()+repr(code.co_names).encode()
    return hashlib.sha256(data).hexdigest()


def run_suite(_):
    p=get_policy()
    scored=benchmark(p,get_rule())
    rec={'time_utc':datetime.now(timezone.utc).isoformat(timespec='seconds'),
         'policy':asdict(p),'score':scored['score'],'percent':scored['percent'],
         'counts':scored['counts'],'outcomes':{k:v.outcome for k,v in scored['results'].items()},
         'advanced':scored['advanced'].outcome,'custom_rule_enabled':bool(use_rule.value),
         'rule_fingerprint':rule_fingerprint()}
    suite_runs.append(rec)
    suite_view.value=render_score(scored)


def coding_status(policy):
    """The student's OWN rule must catch the hidden attack, not another configured limit.

    The same rule must permit an ordinary large supplier payment.
    """
    if not use_rule.value:
        return False, 'Enable your coding-exercise rule in Policy.', None, None
    fn=globals().get('student_rule')
    if not callable(fn):
        return False, 'Run the final Python coding cell first.', None, None
    attack=evaluate(policy,BY_KEY['collusion'],fn)
    legit=evaluate(policy,BY_KEY['large_payment'],fn)
    rule_stopped=attack.why=='Custom risk control' and attack.outcome in ('REVIEW','BLOCKED')
    clean_legit=(legit.outcome=='EXECUTED' and
                 any(s['stage']=='Student custom rule' and s['status']=='PASS' for s in legit.trace))
    if not rule_stopped:
        return False, 'Your Python rule must itself flag full-quorum collusion.', attack, legit
    if not clean_legit:
        return False, 'Your rule/policy must still EXECUTE the legitimate $18,000 supplier payment.', attack, legit
    return True, 'Python rule passed both tests.', attack, legit


choose.observe(update_context,names='value')
for ctrl in recovery_widgets:
    ctrl.observe(participant_changed,names='value')
run_btn.on_click(run_incident)
save_note_btn.on_click(save_observation)
suite_btn.on_click(run_suite)
for ctrl in controls:
    ctrl.observe(preview_policy,names='value')
update_context()
preview_policy()
display(tabs)

### Final coding exercise — catch a full-quorum attack

A suspicious **$18,000 after-hours** transfer has **all required signatures**, an approved destination, and a low risk score. Add one **pre-signing rule** that flags it but still permits a genuine daytime supplier payment.

**You only need these fields (no need to read the bank engine above):**

| Input to `student_rule(tx, policy)` | Meaning |
|:--|:--|
| `tx.after_hours` | `True` for an unusual-hour transaction |
| `tx.amount` | USD amount as an integer |
| `tx.risk` | Synthetic monitoring score from 0–100 |
| `tx.allowlisted` | Whether the payee is on the approved list |

**Return:** `"REVIEW"` to pause, `"BLOCK"` to reject, or `None` to leave the other rules in charge. Python hint: `if tx.after_hours and tx.amount >= ...:`. Do **not** edit the hidden engine.

**Steps:** edit and run the next cell → enable **Use my coding-exercise rule** in Policy → rerun **Challenge** → record your results in the Google Form. Your custom rule should flag collusion while permitting the legitimate large payment.


In [ ]:
# FINAL CODING EXERCISE (required)
# Change only student_rule. Available fields are listed above.
# Return "REVIEW", "BLOCK", or None.

def student_rule(tx, policy):
    # TODO: Flag after-hours transfers of $15,000 or more.
    # A real $18,000 daytime supplier payment must still go through.
    return None

# Short self-check: run this cell after editing your rule.
print('Advanced collusion:', evaluate(get_policy(), BY_KEY['collusion'], student_rule).outcome)
print('Legitimate supplier:', evaluate(get_policy(), BY_KEY['large_payment'], student_rule).outcome)
print('Target: REVIEW (collusion) / EXECUTED (supplier), with suitable bank limits')


**Finish:** Record your baseline and best scores, policy settings, predictions and outcomes for the four required incidents, your `student_rule` function, and one remaining security/usability tradeoff **directly in the course Google Form**. The notebook does not export or submit anything.

**Model boundary:** `PENDING` / `REVIEW` never executes a transfer. Recovery and MPC are illustrative; there is no real blockchain transaction.
